In [1]:
import pandas as pd

In [2]:
df = pd.read_parquet(path='../data/preprocess_reviews.gzip')

In [3]:
df.info()

<class 'pandas.DataFrame'>
Index: 1105228 entries, 0 to 1124122
Data columns (total 6 columns):
 #   Column                                   Non-Null Count    Dtype
---  ------                                   --------------    -----
 0   content                                  1105228 non-null  str  
 1   label                                    1105228 non-null  int64
 2   content_with_translate_emojis            1105228 non-null  str  
 3   processed_content                        1105228 non-null  str  
 4   processed_content_with_translate_emojis  1105228 non-null  str  
 5   sentiments                               1105228 non-null  str  
dtypes: int64(1), str(5)
memory usage: 1.0 GB


In [4]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.model_selection import train_test_split, GridSearchCV
import spacy

In [5]:
nlp = spacy.load("fr_core_news_md")

In [6]:
len(nlp.Defaults.stop_words)

507

In [7]:
negation = [
    "ne", "n'", "pas", "jamais", "rien",
    "personne", "aucun", "aucune", "nul",
    "ni", "sans", "guère", "point"
]

affirmation = [
    "oui", "certes", "vraiment", "certainement",
    "sûrement", "effectivement", "absolument",
    "évidemment"
]

stop_words = affirmation + negation

for word in stop_words:
    if word in nlp.Defaults.stop_words:
        nlp.Defaults.stop_words.discard(word)
        nlp.vocab[word].is_stop = False


In [8]:
len(nlp.Defaults.stop_words)

499

In [9]:
stop_words = list(nlp.Defaults.stop_words)

In [10]:
count_vectorizer = CountVectorizer(
    ngram_range=(1, 3),
    stop_words=stop_words,
    min_df=2,
    max_df=0.8,
)

In [11]:
tf_idf_vectorizer = TfidfVectorizer(min_df=2,
                             max_df=0.8,
                             sublinear_tf=True,
                             ngram_range=(1,3),
                             strip_accents='unicode',
                             use_idf=True,
                             stop_words=stop_words)

In [12]:
tf_idf_vectorizer_wo_smooth = TfidfVectorizer(min_df=2,
                             max_df=0.8,
                             sublinear_tf=True,
                             ngram_range=(1,3),
                             strip_accents='unicode',
                             use_idf=True,
                             smooth_idf=False,
                             stop_words=stop_words)

In [13]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

classes = np.unique(df["label"])

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=df["label"]
)

class_weight = dict(zip(classes, weights))

print(class_weight)

{np.int64(0): np.float64(1.575011364818512), np.int64(1): np.float64(12.67230783342506), np.int64(2): np.float64(0.43741246134843265)}


In [14]:
X = df['processed_content']
y = df['label']

In [15]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

In [16]:
len(y_test), len(X_test)

(331569, 331569)

In [17]:
X_test = X_test[len(y_test)//2:]
y_test = y_test[len(y_test)//2:]
X_valid = X_test[:len(y_test)//2]
y_valid = y_test[:len(y_test)//2]

In [18]:
trains_count_vectors=count_vectorizer.fit_transform(X_train)

/home/lip/venv/lib/python3.12/site-packages/sklearn/feature_extraction/text.py:412: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['neuf', 'qu', 'quelqu'] not in stop_words.
  warnings.warn(


In [19]:
trains_tf_idf_vectors=tf_idf_vectorizer.fit_transform(X_train)

/home/lip/venv/lib/python3.12/site-packages/sklearn/feature_extraction/text.py:412: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['ca', 'cinquantieme', 'cinquieme', 'dela', 'deuxieme', 'deuxiemement', 'dixieme', 'douzieme', 'excepte', 'he', 'huitieme', 'neuf', 'neuvieme', 'notres', 'onzieme', 'premiere', 'premierement', 'qu', 'quatrieme', 'quatriemement', 'quelqu', 'septieme', 'sixieme', 'troisieme', 'troisiemement', 've'] not in stop_words.
  warnings.warn(


In [20]:
trains_tf_idf_no_smooth_vectors=tf_idf_vectorizer_wo_smooth.fit_transform(X_train)

In [28]:
val_count_vectors=count_vectorizer.transform(X_valid)

In [29]:
test_count_vectors=count_vectorizer.transform(X_test)

In [30]:
from sklearn import svm
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
clf_svm_ovo = svm.SVC(decision_function_shape='ovo',random_state=42, probability=True, class_weight=class_weight)
clf_svm_ovo.fit(trains_count_vectors, y_train)
clf_svm_ovo_pred = clf_svm_ovo.predict(test_count_vectors)
report = classification_report(y_test, clf_svm_ovo_pred, output_dict=True)
accuracy_score_svm_ovo = accuracy_score(y_test, clf_svm_ovo_pred)
print("report:", report)
print("accuracy:", accuracy_score_svm_ovo)

/home/lip/venv/lib/python3.12/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
